# Banking Customer 360 Analytics

## 03a - SQL Data Integration & KPI Generation

This notebook demonstrates SQL-based data integration using SQLite.

The cleaned banking tables are loaded into a SQLite database. SQL is then used for joins, aggregations, KPI calculations, customer analysis, cross-sell opportunities, and risk flags.

SQLite is used because it is included with Python and does not require a separate database server.


## 1. Load Cleaned Tables

The cleaned CSV files created by `02_data_cleaning.ipynb` will be loaded into SQLite tables.


In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

processed_path = Path("../data/processed")
db_path = Path("../data/banking_customer360.db")
sql_path = Path("../sql")

sql_path.mkdir(exist_ok=True)

tables = [
    "customers",
    "accounts",
    "transactions",
    "loans",
    "customer_service"
]

conn = sqlite3.connect(db_path)

for table in tables:
    df = pd.read_csv(processed_path / f"{table}.csv")
    
    df.to_sql(
        table,
        conn,
        if_exists="replace",
        index=False
    )
    
    print(f"{table:<20} {df.shape}")

customers            (5000, 26)
accounts             (7500, 6)
transactions         (100000, 7)
loans                (2500, 7)
customer_service     (10000, 6)


In [3]:
for table in tables:
    columns = pd.read_sql(
        f"PRAGMA table_info({table})",
        conn
    )["name"].tolist()

    print(f"\n{table}:")
    print(columns)


customers:
['customer_id', 'age', 'gender', 'country', 'occupation', 'annual_income', 'credit_score', 'tenure_years', 'join_date', 'total_transactions', 'total_transaction_value', 'avg_transaction_value', 'last_transaction_date', 'loan_count', 'total_loan_amount', 'late_loan_count', 'default_loan_count', 'complaint_count', 'escalated_count', 'account_count', 'total_balance', 'active_account_count', 'recency_days', 'products_number', 'churn', 'loan_to_income_ratio']

accounts:
['account_id', 'customer_id', 'account_type', 'opening_date', 'status', 'balance']

transactions:
['transaction_id', 'customer_id', 'account_id', 'transaction_date', 'transaction_type', 'amount', 'channel']

loans:
['loan_id', 'customer_id', 'loan_type', 'loan_amount', 'interest_rate', 'start_date', 'repayment_status']

customer_service:
['interaction_id', 'customer_id', 'interaction_date', 'complaint_type', 'resolution_status', 'resolution_days']


## Build Customer 360 with SQL

Each banking table is first aggregated to one row per customer.

The aggregated tables are then joined to the customer master table.

This prevents duplicate rows and produces one final record per customer.

In [5]:
customer_360_sql_query = """
WITH account_summary AS (
    SELECT
        customer_id,
        COUNT(DISTINCT account_id) AS sql_account_count,
        SUM(balance) AS sql_total_balance,
        AVG(balance) AS sql_avg_balance,
        COUNT(DISTINCT account_type) AS sql_product_holding_count
    FROM accounts
    GROUP BY customer_id
),

transaction_summary AS (
    SELECT
        customer_id,
        COUNT(transaction_id) AS sql_total_transactions,
        SUM(amount) AS sql_total_transaction_value,
        AVG(amount) AS sql_avg_transaction_value
    FROM transactions
    GROUP BY customer_id
),

loan_summary AS (
    SELECT
        customer_id,
        COUNT(DISTINCT loan_id) AS sql_loan_count,
        SUM(loan_amount) AS sql_total_loan_amount,
        AVG(interest_rate) AS sql_avg_interest_rate
    FROM loans
    GROUP BY customer_id
),

service_summary AS (
    SELECT
        customer_id,
        COUNT(interaction_id) AS sql_service_interactions,
        AVG(resolution_days) AS sql_avg_resolution_days
    FROM customer_service
    GROUP BY customer_id
)

SELECT
    c.customer_id,
    c.annual_income,

    COALESCE(a.sql_account_count, 0) AS sql_account_count,
    COALESCE(a.sql_total_balance, 0) AS sql_total_balance,
    COALESCE(a.sql_avg_balance, 0) AS sql_avg_balance,
    COALESCE(a.sql_product_holding_count, 0) AS sql_product_holding_count,

    COALESCE(t.sql_total_transactions, 0) AS sql_total_transactions,
    COALESCE(t.sql_total_transaction_value, 0) AS sql_total_transaction_value,
    COALESCE(t.sql_avg_transaction_value, 0) AS sql_avg_transaction_value,

    COALESCE(l.sql_loan_count, 0) AS sql_loan_count,
    COALESCE(l.sql_total_loan_amount, 0) AS sql_total_loan_amount,
    COALESCE(l.sql_avg_interest_rate, 0) AS sql_avg_interest_rate,

    COALESCE(s.sql_service_interactions, 0) AS sql_service_interactions,
    COALESCE(s.sql_avg_resolution_days, 0) AS sql_avg_resolution_days,

    CASE
        WHEN c.annual_income > 0
        THEN COALESCE(l.sql_total_loan_amount, 0) / c.annual_income
        ELSE 0
    END AS sql_loan_to_income_ratio

FROM customers c

LEFT JOIN account_summary a
    ON c.customer_id = a.customer_id

LEFT JOIN transaction_summary t
    ON c.customer_id = t.customer_id

LEFT JOIN loan_summary l
    ON c.customer_id = l.customer_id

LEFT JOIN service_summary s
    ON c.customer_id = s.customer_id
"""

In [6]:
# Execute SQL
customer_360_sql = pd.read_sql(
    customer_360_sql_query,
    conn
)

print("Shape:", customer_360_sql.shape)

print(
    "Duplicate customer IDs:",
    customer_360_sql["customer_id"].duplicated().sum()
)

customer_360_sql.head()

Shape: (5000, 15)
Duplicate customer IDs: 0


,customer_id,annual_income,sql_account_count,sql_total_balance,sql_avg_balance,sql_product_holding_count,sql_total_transactions,sql_total_transaction_value,sql_avg_transaction_value,sql_loan_count,sql_total_loan_amount,sql_avg_interest_rate,sql_service_interactions,sql_avg_resolution_days,sql_loan_to_income_ratio
0,C00001,25700,4,"337,661.45","84,415.36",2,22,"5,140.08",233.64,0,0.00,0.00,1,8.00,0.00
1,C00002,99800,1,"20,846.07","20,846.07",1,21,"6,220.92",296.23,0,0.00,0.00,2,11.50,0.00
2,C00003,102200,2,"46,466.48","23,233.24",2,16,"4,599.73",287.48,1,"262,227.49",9.12,1,6.00,2.57
3,C00004,48500,1,"208,178.52","208,178.52",1,15,"3,998.08",266.54,0,0.00,0.00,1,2.00,0.00
4,C00005,75900,2,"99,225.27","49,612.64",1,19,"7,296.95",384.05,0,0.00,0.00,1,8.00,0.00


In [7]:
# Save SQL Customer 360
customer_360_sql.to_sql(
    "customer_360_sql",
    conn,
    if_exists="replace",
    index=False
)

customer_360_sql.to_csv(
    "../data/final/customer_360_sql.csv",
    index=False
)

print("SQL Customer 360 saved successfully")

SQL Customer 360 saved successfully


## SQL KPI Analysis

SQL is used to calculate key banking KPIs from the SQL Customer 360 table.


In [8]:
kpi_query = """
SELECT
    COUNT(*) AS total_customers,

    ROUND(SUM(sql_total_balance), 2)
        AS total_customer_balance,

    ROUND(AVG(sql_total_balance), 2)
        AS average_customer_balance,

    SUM(sql_total_transactions)
        AS total_transactions,

    ROUND(SUM(sql_total_transaction_value), 2)
        AS total_transaction_value,

    ROUND(SUM(sql_total_loan_amount), 2)
        AS total_loan_amount,

    ROUND(AVG(sql_product_holding_count), 2)
        AS average_products,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN sql_loan_count > 0 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS loan_penetration_pct

FROM customer_360_sql
"""

sql_kpis = pd.read_sql(kpi_query, conn)

sql_kpis

,total_customers,total_customer_balance,average_customer_balance,total_transactions,total_transaction_value,total_loan_amount,average_products,loan_penetration_pct
0,5000,"429,675,148.55","85,935.03",100000,"29,633,739.80","945,526,046.31",1.16,39.24


In [9]:
# Product Holding Count
product_query = """
SELECT
    sql_product_holding_count AS products_held,
    COUNT(*) AS customers,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS customer_percentage
FROM customer_360_sql
GROUP BY sql_product_holding_count
ORDER BY products_held
"""

product_holding = pd.read_sql(
    product_query,
    conn
)

product_holding

,products_held,customers,customer_percentage
0,0,1122,22.44
1,1,2277,45.54
2,2,1311,26.22
3,3,271,5.42
4,4,19,0.38


In [10]:
# Channel Preference
channel_query = """
SELECT
    channel,
    COUNT(*) AS transactions,
    ROUND(SUM(amount), 2) AS total_value,
    ROUND(AVG(amount), 2) AS average_value
FROM transactions
GROUP BY channel
ORDER BY transactions DESC
"""

channel_preference = pd.read_sql(
    channel_query,
    conn
)

channel_preference

,channel,transactions,total_value,average_value
0,Mobile,40086,"11,881,977.37",296.41
1,Online,30087,"8,923,412.01",296.59
2,ATM,19965,"5,940,174.44",297.53
3,Branch,9862,"2,888,175.98",292.86


In [12]:
# Monthly Transaction Trend - transaction trend / seasonality
monthly_query = """
SELECT
    strftime('%Y-%m', transaction_date) AS month,
    COUNT(*) AS transactions,
    ROUND(SUM(amount), 2) AS total_value
FROM transactions
WHERE transaction_date IS NOT NULL
GROUP BY month
ORDER BY month
"""

monthly_trend = pd.read_sql(
    monthly_query,
    conn
)

monthly_trend

,month,transactions,total_value
0,2025-01,8373,"2,458,029.22"
1,2025-02,7680,"2,277,370.96"
2,2025-03,8515,"2,503,891.13"
3,2025-04,8332,"2,456,636.56"
4,2025-05,8455,"2,504,956.68"
5,2025-06,8155,"2,442,658.97"
6,2025-07,8578,"2,574,138.97"
7,2025-08,8464,"2,484,014.77"
8,2025-09,8113,"2,402,499.13"
9,2025-10,8503,"2,533,894.51"


In [13]:
# Export SQL Results
export_dir = Path("../outputs/exports")

export_dir.mkdir(
    parents=True,
    exist_ok=True
)

sql_kpis.to_csv(
    export_dir / "sql_kpis.csv",
    index=False
)

product_holding.to_csv(
    export_dir / "sql_product_holding.csv",
    index=False
)

channel_preference.to_csv(
    export_dir / "sql_channel_preference.csv",
    index=False
)

monthly_trend.to_csv(
    export_dir / "sql_monthly_trend.csv",
    index=False
)

print("SQL analysis exports saved successfully")

SQL analysis exports saved successfully


In [14]:
conn.close()

print("SQL database connection closed")
print("SQL integration COMPLETE")

SQL database connection closed
SQL integration COMPLETE


In [15]:
# Create SQL Query Script
sql_script = """
-- Banking Customer 360 Analytics
-- SQL Integration and Business Analysis

-- 1. Customer 360 aggregation

WITH account_summary AS (
    SELECT
        customer_id,
        COUNT(DISTINCT account_id) AS account_count,
        SUM(balance) AS total_balance,
        AVG(balance) AS average_balance,
        COUNT(DISTINCT account_type) AS products_held
    FROM accounts
    GROUP BY customer_id
),

transaction_summary AS (
    SELECT
        customer_id,
        COUNT(transaction_id) AS total_transactions,
        SUM(amount) AS total_transaction_value,
        AVG(amount) AS average_transaction_value
    FROM transactions
    GROUP BY customer_id
),

loan_summary AS (
    SELECT
        customer_id,
        COUNT(DISTINCT loan_id) AS loan_count,
        SUM(loan_amount) AS total_loan_amount,
        AVG(interest_rate) AS average_interest_rate
    FROM loans
    GROUP BY customer_id
),

service_summary AS (
    SELECT
        customer_id,
        COUNT(interaction_id) AS service_interactions,
        AVG(resolution_days) AS average_resolution_days
    FROM customer_service
    GROUP BY customer_id
)

SELECT
    c.customer_id,
    c.annual_income,

    COALESCE(a.account_count, 0) AS account_count,
    COALESCE(a.total_balance, 0) AS total_balance,
    COALESCE(a.average_balance, 0) AS average_balance,
    COALESCE(a.products_held, 0) AS products_held,

    COALESCE(t.total_transactions, 0) AS total_transactions,
    COALESCE(t.total_transaction_value, 0) AS total_transaction_value,
    COALESCE(t.average_transaction_value, 0) AS average_transaction_value,

    COALESCE(l.loan_count, 0) AS loan_count,
    COALESCE(l.total_loan_amount, 0) AS total_loan_amount,
    COALESCE(l.average_interest_rate, 0) AS average_interest_rate,

    COALESCE(s.service_interactions, 0) AS service_interactions,
    COALESCE(s.average_resolution_days, 0) AS average_resolution_days

FROM customers c
LEFT JOIN account_summary a
    ON c.customer_id = a.customer_id
LEFT JOIN transaction_summary t
    ON c.customer_id = t.customer_id
LEFT JOIN loan_summary l
    ON c.customer_id = l.customer_id
LEFT JOIN service_summary s
    ON c.customer_id = s.customer_id;


-- 2. Channel analysis

SELECT
    channel,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_value,
    ROUND(AVG(amount), 2) AS average_value
FROM transactions
GROUP BY channel
ORDER BY transaction_count DESC;


-- 3. Monthly transaction trend

SELECT
    strftime('%Y-%m', transaction_date) AS month,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_value
FROM transactions
GROUP BY month
ORDER BY month;


-- 4. Product holding analysis

SELECT
    COUNT(DISTINCT account_type) AS products_held,
    COUNT(DISTINCT customer_id) AS customers
FROM accounts
GROUP BY customer_id
ORDER BY products_held;


-- 5. Loan customer penetration

SELECT
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN loan_count > 0 THEN 1
            ELSE 0
        END
    ) AS loan_customers,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN loan_count > 0 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS loan_penetration_pct
FROM (
    SELECT
        customer_id,
        COUNT(loan_id) AS loan_count
    FROM loans
    GROUP BY customer_id
) loan_summary;
"""

sql_file = Path("../sql/customer_360_queries.sql")

sql_file.write_text(
    sql_script,
    encoding="utf-8"
)

print("SQL script saved successfully:", sql_file)

SQL script saved successfully: ..\sql\customer_360_queries.sql


In [16]:
conn.close()

print("SQLite connection closed.")
print("03a SQL Integration completed successfully.")

SQLite connection closed.
03a SQL Integration completed successfully.
